# Sales Prediction for Big Mart Outlets

The BigMart Sales project focuses on analyzing retail data to understand the key factors influencing product sales across multiple outlets and to develop predictive and analytical insights. The dataset consists of product-level and outlet-level attributes such as item price (MRP), visibility, category, outlet type, and sales.This calls for brushing up my EDA skills and also makes me think deeper on various features and imputations.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings;
warnings.filterwarnings('ignore')
sns.set_style("white")
%matplotlib inline

In [156]:
#Reading the training and testing dataset

df = pd.read_csv('./data/train_bigmart.csv')
df_test = pd.read_csv('./data/test_bigmart.csv')

df_copy = df.copy()
df_test_copy = df_test.copy()
df.head()



,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [ ]:
print(f"{df.shape = }, {df_test.shape=}")

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

## Observations:


  1. Item_Weight — Only 7,060 of 8,523 rows have values (~1,463 missing, ~17%). Range (4.56–21.35) and mean/std look reasonable, so missing values likely need imputation (e.g., by item identifier/type group mean).
  2. Item_Visibility — Minimum is 0.0, which is physically implausible (every shelved item has some visibility). This is likely a data-entry artifact / disguised missing value and should be treated (e.g., replace 0 with NaN
  and impute). The mean (0.066) is noticeably higher than the median (0.054), and the max (0.33) is far beyond the 75th percentile (0.095) — suggests a right-skewed distribution with potential outliers.
  3. Item_MRP — No missing values. Mean (141.0) ≈ median (143.0), fairly symmetric distribution, std ~62. Range (31.3–266.9) seems plausible for product prices, no obvious outliers.
  4. Outlet_Establishment_Year — Ranges 1985–2009, treated as numeric but is really ordinal/categorical. Useful to convert into "Outlet_Age" (e.g., current_year − establishment_year) as a feature for modeling.
  5. Item_Outlet_Sales (target variable) — Mean (2181) > median (1794), and max (13,087) is ~4x the 75th percentile (3101) → strong right skew, likely with high-value outliers. A log-transform may help if this is used as a regression target, and models should be evaluated with this skew in mind.

- Item_Visibility and Item_Outlet_Sales both show skew/outlier signals worth visualizing (histograms/boxplots) before modeling.
  - Categorical columns (Item_Type, Outlet_Type, Outlet_Size, etc.)

In [ ]:
df_test.describe()

#### Merge data

In [157]:
data = pd.concat([df,df_test])
data.shape

(14204, 12)

In [ ]:
print(data['Item_Outlet_Sales'].isna().sum() == df_test.shape[0])

### Observations:
It does not make sense to merge test as those do not have the Target variable. The only reason to use the merged data is to analyse the other variables


In [ ]:

plt.hist(df['Item_Outlet_Sales'], bins = 20, color = 'pink')
plt.title('Target Variable')
plt.xlabel('Item Outlet Sales')
plt.ylabel('count')
plt.show()

In [ ]:
def plot_column_distributions(df, columns=None, max_categories=20):
    """
    For each column, print value_counts and show a bar or hist plot.
    Columns with more than max_categories unique values get a histogram;
    lower-cardinality columns get a bar chart.
    """
    if columns is None:
        columns = df.columns.tolist()

    for col in columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts())
        print(df[col].value_counts(normalize=True).round(4).to_string())

        fig, ax = plt.subplots(figsize=(10, 4))
        vc = df[col].value_counts()

        if df[col].nunique() > max_categories:
            vc.plot.hist(ax=ax, bins=30)
        else:
            vc.plot.bar(ax=ax)

        ax.set_title(f'Distribution of {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:
# Plot all columns in df — or pass a list to limit scope, e.g.:
plot_column_distributions(df, columns=['Item_Fat_Content'])
# plot_column_distributions(df)


In [ ]:
plot_column_distributions(df, columns=['Item_Type'])

In [ ]:
plot_column_distributions(df, columns=['Outlet_Identifier'])

In [ ]:
plot_column_distributions(df, columns=['Outlet_Size'])

In [ ]:
plot_column_distributions(df, columns=['Outlet_Location_Type'])

In [ ]:
plot_column_distributions(df, columns=['Outlet_Type'])

In [ ]:
Item_Type = pd.crosstab(df['Item_Type'], df['Item_Fat_Content'])
Item_Type.div(Item_Type.sum(1).astype(float), axis=0).plot(kind="bar", stacked=True, figsize=(13, 13))

In [ ]:
data.apply(lambda x : len(x.unique()))

### EDA - preprocess, before train-test split

Some of the data have data entry errors. For example, item_fat_content has value Regular and reg. This should be cleaned up across. Such imputations need not wait till split.


In [ ]:
data['Item_Identifier'].apply(lambda x : x[0:2]).unique()

### Observations:

['FD', 'DR', 'NC'] could map to Food, Drink and Non-Consumable



In [158]:
df['Item_Identifier_Type'] = df['Item_Identifier'].apply(lambda x: x[0:2])
df_test['Item_Identifier_Type'] = df_test['Item_Identifier'].apply(lambda x: x[0:2])
df['Item_Identifier_Type'].unique()

<ArrowStringArray>
['FD', 'DR', 'NC']
Length: 3, dtype: str

### Since Non Consumables cannot be low fat or regular so  will create new var in item fat content for non consumable item identifier

In [159]:
df.loc[df['Item_Identifier_Type']=='NC','Item_Fat_Content']='No Fat'
df_test.loc[df_test['Item_Identifier_Type']=='NC','Item_Fat_Content']='No Fat'


In [ ]:
df.head()

In [ ]:
cat_features = df.select_dtypes(include=['object','category']).columns.tolist()
cat_features


In [ ]:
df['Item_Fat_Content'].value_counts()

we can see that  that there is mistake in collection the data for  Item_Fat_Content .

For Low Fat they have used three symbols which are Low Fat, LF an low fat so we will replace that values by original values that areLow fat and regular

In [160]:
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace('LF','Low Fat')
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace('low fat','Low Fat')
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace('reg','Regular')

df_test['Item_Fat_Content'] = df_test['Item_Fat_Content'].replace('LF','Low Fat')
df_test['Item_Fat_Content'] = df_test['Item_Fat_Content'].replace('low fat','Low Fat')
df_test['Item_Fat_Content'] = df_test['Item_Fat_Content'].replace('reg','Regular')

### Filling the missing values

In [ ]:
df.isnull().sum()

In [ ]:
#An item in the shelf cannot have Item Visibility of 0
df['Item_Visibility'].value_counts()


In [161]:
df['Item_Visibility'] = df['Item_Visibility'].replace(0,np.nan)
df_test['Item_Visibility'] = df_test['Item_Visibility'].replace(0,np.nan)

In [ ]:
df['Outlet_Size'].isna().sum()

### Outlet size cannot be na

In [162]:
df['Outlet_Size'] = df['Outlet_Size'].fillna('Unknown')
df_test['Outlet_Size'] = df_test['Outlet_Size'].fillna('Unknown')

In [ ]:
# plt.figure(figsize=(8,6))
# sns.barplot(df['Outlet_Size'],df['Item_Outlet_Sales'])
# plt.show()

plt.figure(figsize=(8,6))
sns.barplot(x='Outlet_Size', y='Item_Outlet_Sales', data=df)
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.barplot(x='Outlet_Size', y='Item_Visibility', data=df)
plt.show()

From above two graph we can conclude as follows:

Graph 1 : Generally there is low outlet sales in stores having outlet size small and unknown . Also store having small outlet size resembles the store having unknown outlet size

Graph 1 : Generally there is high visibility in stores having outlet size small and unknown . Also store having small outlet size resembles the store having unknown outlet size

In [ ]:
a=pd.crosstab(df['Outlet_Size'],df['Outlet_Type'])
a.plot(kind='bar')
plt.show()

Above graph also tells us that outlet having small size resembles the outlet having unknown size.Also outlet having smaller size are supermaket type 1 or grocery store

In [ ]:
# df1=df[df['Outlet_Size'] == 'Unknown']
# df1.head()
df[df['Outlet_Size'] == 'Unknown']

In [ ]:
# df1['Outlet_Type'].unique()

This dataframe also supports the previous reason that outlet having unknown size are of type grocery store or supermarket type 1

So i'll fill the missing values of outlet size by small as there are enough evidence

In [163]:
df['Outlet_Size'] = df['Outlet_Size'].replace('Unknown','Small')
df_test['Outlet_Size'] = df_test['Outlet_Size'].replace('Unknown','Small')

In [ ]:
target = 'Item_Outlet_Sales'

In [ ]:
traincols = set(df.columns)
testcols=set(df_test.columns)

assert (

    set(testcols).issubset(traincols)
    and len(df.columns) - len(df_test.columns) == 1
    and (traincols - testcols) == {target}
)

In [ ]:
df.isnull().sum()

In [164]:
seed=42
global_test_size=.2

In [165]:
from sklearn.model_selection import train_test_split

X= df.drop(columns=[target])
y=df[target]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=global_test_size,random_state=seed)

X_train_copy = X_train.copy()
X_test_copy = X_test.copy()

### Now Impute missing features using X_Train

In [ ]:
df['Item_Visibility'].isna().sum()

In [ ]:
df[df['Item_Visibility'].isna()]

Why Item_Weight is a Good Choice


Highest Correlation with Missingness:

From your earlier analysis, Item_Weight had the highest percentage of NaN values (~17.11%) in the rows where Item_Visibility is NaN.
This suggests that Item_Weight and Item_Visibility might be related (e.g., heavier items may have different visibility patterns).

Domain Logic:

In retail, item weight can influence shelf visibility (e.g., heavier items might be placed differently).
Grouping by Item_Weight (or Item_Identifier + Item_Weight) can capture item-specific visibility trends.






In [ ]:
df['Item_Weight'].isnull().sum()

Use a hierarchy of grouping to handle cases where Item_Weight is missing:

### Impute Item_Weight and Visibility

In [ ]:
df[df['Item_Weight'].isna()]

In [166]:
vis_mean_by_id = X_train.groupby('Item_Identifier')['Item_Visibility'].mean()
vis_global_mean = X_train['Item_Visibility'].mean()  # The absolute safety net
for d in (X_train, X_test, df_test):
    d['Item_Visibility'] = d['Item_Visibility'].fillna(d['Item_Identifier'].map(vis_mean_by_id))
    # Step 2: Catch any remaining NaNs with the global training mean
    d['Item_Visibility'] = d['Item_Visibility'].fillna(vis_global_mean)

item_weight_mean_by_id = X_train.groupby('Item_Identifier')['Item_Weight'].mean()
item_weight_median_by_type = X_train.groupby('Item_Type')['Item_Weight'].median()
for d in (X_train, X_test, df_test):
    d['Item_Weight'] = d['Item_Weight'].fillna(d['Item_Identifier'].map(item_weight_mean_by_id))
    d['Item_Weight'] = d['Item_Weight'].fillna(d['Item_Type'].map(item_weight_median_by_type))

In [167]:
X_train['Item_Weight'].isna().sum()

np.int64(0)

In [ ]:
assert X_train['Item_Visibility'].isna().sum() == X_train['Item_Visibility'].isnull().sum()

### Preprocessing is completed. This should be a good placeholder for saving files for pyod notebook

In [168]:
precleaned = pd.concat([
    X_train.assign(**{target: y_train}),
    X_test.assign(**{target: y_test}),
    ]).sort_index()
precleaned.to_csv('data/train_precleaned.csv',index=False)
print(f"Saved precleaned data to data/train_precleaned.csv")

Saved precleaned data to data/train_precleaned.csv


In [169]:
df_test.to_csv('data/test_precleaned.csv',index=False)
print(f"Saved precleaned data to data/test_precleaned.csv")

Saved precleaned data to data/test_precleaned.csv


### Outlier handling — fit cap threshold on X_train only; X_test/df_test untouched

In [ ]:
## Several techiniques like z-score, iqr or using std deviation

In [170]:
q1, q3 = X_train['Item_Visibility'].quantile([.25,.75])
iqr = q3 -q1
upper_bound = q3 + 1.5 * iqr
replacement = X_train['Item_Visibility'].mean()

for d in (X_train,X_test,df_test):
    d['Item_Visibility'] = d['Item_Visibility'].apply(lambda  x : replacement if x > upper_bound else x)


In [171]:
X_Train_1 =X_train.copy()
X_test1= X_test.copy()
df_test1=df_test.copy()

In [172]:
# X_train=X_Train_1.copy()
# X_test=X_test1.copy()
# df_test=df_test1.copy()

In [173]:
def get_datasets():
    return (X_train,X_test,df_test)

### Additional New Features post split

In [174]:
mrp_bin_labels = ['cheap', 'affordable', 'slightly expensive', 'expensive']
weight_bin_labels = ['vlight', 'light', 'moderate', 'heavy']
X_train['MRP_bin'], mrp_bins = pd.qcut(X_train['Item_MRP'], q=4, labels=mrp_bin_labels, retbins=True)
X_train['Weight_bin'], weight_bins = pd.qcut(X_train['Item_Weight'], q=4, labels=weight_bin_labels, retbins=True)
mrp_bins[0], mrp_bins[-1] = -np.inf, np.inf
weight_bins[0], weight_bins[-1] = -np.inf, np.inf

for d in (X_test, df_test):
    d['MRP_bin'] = pd.cut(d['Item_MRP'], bins=mrp_bins, labels=mrp_bin_labels)
    d['Weight_bin'] = pd.cut(d['Item_Weight'], bins=weight_bins, labels=weight_bin_labels)

In [175]:
calorie_map = {
    'Dairy': 46,
    'Soft Drinks': 51,
    'Meat': 143,
    'Fruits and Vegetables': 65,
    'Baking Goods': 140,
    'Snack Foods': 475,
    'Frozen Foods': 50,
    'Breakfast': 350,
    'Hard Drinks': 250,
    'Canned': 80,
    'Starchy Foods': 90,
    'Seafood': 204,
    'Breads': 250,
}

item_frequency_map={'Dairy':'D','Meat':'S','Fruits and Vegetables':'D','Breakfast':'S','Breads':'S','Starchy Foods':'S','Seafood':'S','Soft Drinks':'S','Household':'D','Baking Goods':'S','Snack Foods':'D','Frozen Foods':'D','Hard Drinks':'S','Canned':'D','Health and Hygiene':'S','Others':'S'}

In [176]:
    # calorie_map / item_frequency_map are fixed domain dicts (not fit from data) -- apply to all three directly
for d in (X_train, X_test, df_test):
    d['Calorie_Count_per_100g'] = d['Item_Type'].map(calorie_map).fillna(0)
    d['Calorie_Count_per_givenwt'] = d['Calorie_Count_per_100g'] / 100 * d['Item_Weight']
    d['Item_Frequency'] = d['Item_Type'].map(item_frequency_map)
    d['Outlet_Existence'] = 2020 - d['Outlet_Establishment_Year']
    d['Outlet_Status'] = (d['Outlet_Establishment_Year'] > 2000).astype(int)

In [177]:
X_train.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Identifier_Type', 'MRP_bin', 'Weight_bin',
       'Calorie_Count_per_100g', 'Calorie_Count_per_givenwt', 'Item_Frequency',
       'Outlet_Existence', 'Outlet_Status'],
      dtype='str')

### Skewness

Diagnose on X_train only

In [178]:
print(
    X_train[['Item_Weight','Item_MRP','Item_Visibility','Calorie_Count_per_givenwt']].skew()
)

Item_Weight                  0.069549
Item_MRP                     0.106298
Item_Visibility              0.768092
Calorie_Count_per_givenwt    1.992158
dtype: float64


In [179]:
## Lets apply for skew > 1

# log/sqrt are parameter-free, so safe to apply identically everywhere once the check says they're needed
for d in get_datasets():
    d['log_visibility'] = np.log(d['Item_Visibility'])
    d['log_Calorie'] = np.sqrt(d['Calorie_Count_per_givenwt'])

In [180]:
X_train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,...,Item_Identifier_Type,MRP_bin,Weight_bin,Calorie_Count_per_100g,Calorie_Count_per_givenwt,Item_Frequency,Outlet_Existence,Outlet_Status,log_visibility,log_Calorie
549,FDW44,9.500,Regular,0.035206,Fruits and Vegetables,171.3448,OUT049,1999,Medium,Tier 1,...,FD,slightly expensive,light,65.0,6.17500,D,21,0,-3.346543,2.484955
7757,NCF54,18.000,No Fat,0.047473,Household,170.5422,OUT045,2002,Small,Tier 2,...,NC,slightly expensive,heavy,0.0,0.00000,D,18,1,-3.047591,0.000000
764,FDY03,17.600,Regular,0.076122,Meat,111.7202,OUT046,1997,Small,Tier 1,...,FD,affordable,heavy,143.0,25.16800,S,23,0,-2.575420,5.016772
6867,FDQ20,8.325,Low Fat,0.029845,Fruits and Vegetables,41.6138,OUT045,2002,Small,Tier 2,...,FD,cheap,vlight,65.0,5.41125,D,18,1,-3.511730,2.326209
2716,FDP34,12.850,Low Fat,0.137228,Snack Foods,155.5630,OUT046,1997,Small,Tier 1,...,FD,slightly expensive,moderate,475.0,61.03750,D,23,0,-1.986113,7.812650


### Label Encoding

In [181]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

#Label Encoding Types
for col in ['Item_Identifier_Type', 'Item_Type', 'Outlet_Type', 'Outlet_Identifier']:
    le= LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    df_test[col] = le.transform(df_test[col])
    label_encoders[col] = le

#Ordinal Map Encoding types
ordinal_maps = {
    'Outlet_Size': {'Small': 0, 'Medium': 1, 'High': 2},
    'MRP_bin': {'cheap': 0, 'affordable': 1, 'slightly expensive': 2, 'expensive': 3},
    'Weight_bin': {'vlight': 0, 'light': 1, 'moderate': 2, 'heavy': 3},
    'Outlet_Location_Type': {'Tier 3': 0, 'Tier 2': 1, 'Tier 1': 2},
    'Item_Fat_Content': {'No Fat': 0, 'Low Fat': 1, 'Regular': 2},
    'Item_Frequency': {'D': 1, 'S': 0},
}

for col, mapping in ordinal_maps.items():
    for d in get_datasets():
        d[col] = d[col].map(mapping)

#Rank-encoding for features with years
# Outlet_Existence and Outlet_Establishment_Year are rank-encodings of the same years
# -> fit the rank on X_train's years only, reuse on X_test/df_test
year_rank = {year: rank for rank, year in enumerate(sorted(X_train['Outlet_Establishment_Year'].unique()))}
existence_rank = {2020 - year: rank for year, rank in year_rank.items()}

for d in get_datasets():
    d['Outlet_Existence'] = d['Outlet_Existence'].map(existence_rank)
    d['Outlet_Establishment_Year'] = d['Outlet_Establishment_Year'].map(year_rank)

X_train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,...,Item_Identifier_Type,MRP_bin,Weight_bin,Calorie_Count_per_100g,Calorie_Count_per_givenwt,Item_Frequency,Outlet_Existence,Outlet_Status,log_visibility,log_Calorie
549,FDW44,9.500,2,0.035206,6,171.3448,9,4,1,2,...,1,2,1,65.0,6.17500,1,4,0,-3.346543,2.484955
7757,NCF54,18.000,0,0.047473,9,170.5422,7,5,0,1,...,2,2,3,0.0,0.00000,1,5,1,-3.047591,0.000000
764,FDY03,17.600,2,0.076122,10,111.7202,8,2,0,2,...,1,1,3,143.0,25.16800,0,2,0,-2.575420,5.016772
6867,FDQ20,8.325,1,0.029845,6,41.6138,7,5,0,1,...,1,0,0,65.0,5.41125,1,5,1,-3.511730,2.326209
2716,FDP34,12.850,1,0.137228,13,155.5630,8,2,0,2,...,1,2,2,475.0,61.03750,1,2,0,-1.986113,7.812650


### Scale — fit StandardScaler on X_train only

In [182]:
from sklearn.preprocessing import StandardScaler

scale_features = ['Item_Weight', 'Item_MRP', 'log_visibility', 'log_Calorie']
# raw skewed duplicates dropped per the earlier skew check
scaler = StandardScaler()
X_train[scale_features] = scaler.fit_transform(X_train[scale_features])
X_test[scale_features] = scaler.transform(X_test[scale_features])
df_test[scale_features] = scaler.transform(df_test[scale_features])

In [183]:
X_train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,...,Item_Identifier_Type,MRP_bin,Weight_bin,Calorie_Count_per_100g,Calorie_Count_per_givenwt,Item_Frequency,Outlet_Existence,Outlet_Status,log_visibility,log_Calorie
549,FDW44,-0.736531,2,0.035206,6,0.470709,9,4,1,2,...,1,2,1,65.0,6.17500,1,4,0,-0.532978,-0.331257
7757,NCF54,1.096585,0,0.047473,9,0.457877,7,5,0,1,...,2,2,3,0.0,0.00000,1,5,1,-0.141163,-1.338259
764,FDY03,1.010321,2,0.076122,10,-0.482625,8,2,0,2,...,1,1,3,143.0,25.16800,0,2,0,0.477679,0.694736
6867,FDQ20,-0.989933,1,0.029845,6,-1.603553,7,5,0,1,...,1,0,0,65.0,5.41125,1,5,1,-0.749477,-0.395587
2716,FDP34,-0.014068,1,0.137228,13,0.218375,8,2,0,2,...,1,2,2,475.0,61.03750,1,2,0,1.250044,1.827737


In [184]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 6818 entries, 549 to 7270
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Item_Identifier            6818 non-null   str     
 1   Item_Weight                6818 non-null   float64 
 2   Item_Fat_Content           6818 non-null   int64   
 3   Item_Visibility            6818 non-null   float64 
 4   Item_Type                  6818 non-null   int64   
 5   Item_MRP                   6818 non-null   float64 
 6   Outlet_Identifier          6818 non-null   int64   
 7   Outlet_Establishment_Year  6818 non-null   int64   
 8   Outlet_Size                6818 non-null   int64   
 9   Outlet_Location_Type       6818 non-null   int64   
 10  Outlet_Type                6818 non-null   int64   
 11  Item_Identifier_Type       6818 non-null   int64   
 12  MRP_bin                    6818 non-null   category
 13  Weight_bin                 6818 non-null   cate

### ANOVA / feature selection — runs on X_train, y_train only

  Why ANOVA is the right test here (categorical predictor → continuous target)
  - The question being asked is: "Does knowing the category change the average value of the target?" That's a classic
  one-way ANOVA setup — one categorical factor, one continuous outcome.
  - Why not a t-test? A t-test only compares two groups. Most of these categorical features (Outlet_Type, Item_Type,
  Outlet_Identifier, etc.) have 3+ levels, so a t-test isn't directly applicable without doing many pairwise
  comparisons (which inflates false-positive risk). ANOVA generalizes the t-test to any number of groups in a single
  test.
  - Why not chi-square? Chi-square tests independence between two categorical variables (via a contingency table of
  counts). It can't be used here because the target (sales) is continuous, not categorical — there's no contingency
  table to build.
  - The F-statistic itself: f_oneway computes the ratio of between-group variance to within-group variance. A large
  F-stat means the differences in average sales across category levels are large relative to the natural variability
  within each level — i.e., the category is a meaningful driver of the target, not just noise. That's exactly what
  you saw with Outlet_Type (F=694) vs Item_Fat_Content (F=0.81, not significant).

  Why Pearson for the continuous features
  For Item_MRP, Item_Visibility, etc., both the feature and target are continuous, so the relevant question becomes
  "is there a linear association between them?" — that's exactly what Pearson's r measures (covariance normalized by
  the standard deviations), with the accompanying p-value testing whether that linear association is distinguishable
  from zero. ANOVA wouldn't apply here since there are no discrete groups to compare.

  One caveat worth knowing
  ANOVA assumes roughly normal residuals and equal variances across groups. With large samples (this dataset has
  thousands of rows per level for most features), the test is fairly robust to non-normality (Central Limit Theorem),
  but if group sizes/variances are very unequal (e.g., Outlet_Identifier levels won't have equal row counts), the
  p-values can be slightly optimistic. If you ever wanted a non-parametric check, kruskal (Kruskal-Wallis) from
  scipy.stats is the rank-based analogue that doesn't assume normality or equal variances — useful as a sanity-check
  cross-validation of the ANOVA results, not a replacement.

In [185]:
# Label/ordinal encoding is a bijective renaming, so grouping by the encoded
# integer codes here is equivalent to grouping by the original category labels.
y_train_series = y_train[target] if isinstance(y_train, pd.DataFrame) else y_train

cat_anova_pr_cols = [
    'Item_Identifier_Type', 'Item_Type', 'Outlet_Type', 'Outlet_Identifier',
    'Outlet_Size', 'MRP_bin', 'Weight_bin', 'Outlet_Location_Type',
    'Item_Fat_Content', 'Item_Frequency', 'Outlet_Status',
    'Outlet_Establishment_Year', 'Outlet_Existence',
]
continuous_cols = [
    c for c in X_train.columns
    if c not in cat_anova_pr_cols and pd.api.types.is_numeric_dtype(X_train[c])
]

continuous_cols

['Item_Weight',
 'Item_Visibility',
 'Item_MRP',
 'Calorie_Count_per_100g',
 'Calorie_Count_per_givenwt',
 'log_visibility',
 'log_Calorie']

In [186]:
non_numeric = [c for c in continuous_cols if not pd.api.types.is_numeric_dtype(X_train[c])]
print(non_numeric)

[]


In [187]:
##dropping as item_identifier_type already captures
X_train = X_train.drop(columns=['Item_Identifier'])
X_test = X_test.drop(columns=['Item_Identifier'])
df_test = df_test.drop(columns=['Item_Identifier'])

In [188]:
traincols = set(X_train.columns)
testcols=set(continuous_cols + cat_anova_pr_cols )

# assert  (traincols == testcols)


In [189]:
X_train.columns

Index(['Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type',
       'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type',
       'Item_Identifier_Type', 'MRP_bin', 'Weight_bin',
       'Calorie_Count_per_100g', 'Calorie_Count_per_givenwt', 'Item_Frequency',
       'Outlet_Existence', 'Outlet_Status', 'log_visibility', 'log_Calorie'],
      dtype='str')

In [190]:
 # 2. Dedup -- pairwise correlation > 0.8, keep whichever side correlates more with target
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
target_corr = X_train.corrwith(y_train_series).abs()

high_corr_pairs = upper.stack()
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.8]

to_drop = set()
for feat1, feat2 in high_corr_pairs.index:
    weaker = feat1 if target_corr[feat1] < target_corr[feat2] else feat2
    to_drop.add(weaker)

print(f"Dropping redundant, weaker-corr-with-target features: {sorted(to_drop)}")

for d in get_datasets():
    d.drop(columns=list(to_drop), inplace=True)


Dropping redundant, weaker-corr-with-target features: ['Calorie_Count_per_100g', 'Item_Weight', 'MRP_bin', 'Outlet_Establishment_Year', 'Outlet_Existence', 'log_Calorie', 'log_visibility']


In [191]:
X_train.columns

Index(['Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP',
       'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Identifier_Type', 'Weight_bin',
       'Calorie_Count_per_givenwt', 'Item_Frequency', 'Outlet_Status'],
      dtype='str')

In [192]:
from scipy.stats import f_oneway, pearsonr

categorical_cols = [
    'Item_Identifier_Type', 'Item_Type', 'Outlet_Type', 'Outlet_Identifier',
    'Outlet_Size', 'MRP_bin', 'Weight_bin', 'Outlet_Location_Type',
    'Item_Fat_Content', 'Item_Frequency', 'Outlet_Status',
    'Outlet_Establishment_Year', 'Outlet_Existence',
]
categorical_cols = [c for c in categorical_cols if c in X_train.columns]
continuous_cols = [c for c in X_train.columns if c not in categorical_cols]

print(f"{categorical_cols = }, {continuous_cols = }")

categorical_cols = ['Item_Identifier_Type', 'Item_Type', 'Outlet_Type', 'Outlet_Identifier', 'Outlet_Size', 'Weight_bin', 'Outlet_Location_Type', 'Item_Fat_Content', 'Item_Frequency', 'Outlet_Status'], continuous_cols = ['Item_Visibility', 'Item_MRP', 'Calorie_Count_per_givenwt']


In [193]:
results = []
for col in categorical_cols:
    groups = [y_train_series[X_train[col] == level] for level in X_train[col].unique()]
    # print(groups)
    f_stat, p_value = f_oneway(*groups)
    results.append({'Feature': col, 'Test': 'ANOVA', 'Statistic': f_stat, 'p_value': p_value})

for col in continuous_cols:
    r, p_value = pearsonr(X_train[col], y_train_series)
    results.append({'Feature': col, 'Test': 'Pearson', 'Statistic': r, 'p_value': p_value})



In [194]:
anova_df = pd.DataFrame(results).sort_values('p_value').reset_index(drop=True)
anova_df

,Feature,Test,Statistic,p_value
0,Outlet_Identifier,ANOVA,232.590830,0.000000e+00
1,Outlet_Type,ANOVA,694.230875,0.000000e+00
2,Item_MRP,Pearson,0.565303,0.000000e+00
3,Outlet_Size,ANOVA,175.330323,5.614245e-75
4,Outlet_Location_Type,ANOVA,41.848905,8.628824e-19
5,Item_Visibility,Pearson,-0.087400,4.858708e-13
6,Item_Frequency,ANOVA,12.112705,5.039297e-04
7,Item_Type,ANOVA,2.149622,6.048495e-03
8,Item_Identifier_Type,ANOVA,4.917658,7.342226e-03
9,Outlet_Status,ANOVA,5.908955,1.508972e-02


In [195]:
print(anova_df)

                      Feature     Test   Statistic       p_value
0           Outlet_Identifier    ANOVA  232.590830  0.000000e+00
1                 Outlet_Type    ANOVA  694.230875  0.000000e+00
2                    Item_MRP  Pearson    0.565303  0.000000e+00
3                 Outlet_Size    ANOVA  175.330323  5.614245e-75
4        Outlet_Location_Type    ANOVA   41.848905  8.628824e-19
5             Item_Visibility  Pearson   -0.087400  4.858708e-13
6              Item_Frequency    ANOVA   12.112705  5.039297e-04
7                   Item_Type    ANOVA    2.149622  6.048495e-03
8        Item_Identifier_Type    ANOVA    4.917658  7.342226e-03
9               Outlet_Status    ANOVA    5.908955  1.508972e-02
10  Calorie_Count_per_givenwt  Pearson    0.027506  2.313278e-02
11           Item_Fat_Content    ANOVA    0.807576  4.459802e-01
12                 Weight_bin    ANOVA    0.453024  7.151644e-01


### Observations:

Strongest predictors (highly significant, large effect)
  - Outlet_Type (F=694.2) and Outlet_Identifier (F=232.6) are by far the dominant categorical drivers of the target.
  Since each outlet has a fixed Type/Size/Location, these three are likely capturing overlapping signal —
  Outlet_Identifier is probably just a finer-grained proxy for Outlet_Type (watch for multicollinearity/redundancy if
  used together in a linear model).
  - Item_MRP (r=0.565) is the strongest continuous predictor — a solid positive correlation, which makes sense since
  sales value scales with item price.
  - Outlet_Size (F=175.3) and Outlet_Location_Type (F=41.8) are also strongly significant, reinforcing that
  where/what kind of store matters more than what item is being sold.

  Moderate/weak but statistically significant
  - Item_Visibility (r=−0.087, p≈0) — weak but significant negative correlation. This is a known counterintuitive
  quirk in this dataset (items with near-zero visibility still sell, possibly staples); effect size is small so
  practical impact is limited.
  - Item_Frequency (p=0.0005), Item_Identifier_Type (p=0.007), Outlet_Status (p=0.015) are statistically significant
  but with small F-stats — real but minor effects, likely add marginal predictive value.
  - Item_Type (F=2.15, p=0.006) — significant only because of the large sample size/many categories; the effect size
  itself is weak, so this feature alone isn't a strong differentiator.
  - Calorie_Count_per_givenwt (r=0.0275, p=0.023) — borderline significant only due to large n; correlation is
  negligible in practical terms. Likely not useful as a standalone feature.

  Not significant — candidates to drop or deprioritize
  - Item_Fat_Content (p=0.446) and Weight_bin (p=0.715) show no meaningful relationship with the target. Unless
  there's a strong domain reason or interaction effect you suspect, these add noise rather than signal.

  Overall takeaway
  Outlet-level features (Type, Identifier, Size, Location) and Item_MRP explain most of the variance; item-level
  categorical attributes (Fat Content, Weight_bin, Item_Type) are weak-to-irrelevant on their own. For feature
  selection, I'd prioritize Outlet_Type/Size/Location_Type + Item_MRP, treat Outlet_Identifier as redundant unless
  store-specific effects matter, and consider dropping Item_Fat_Content/Weight_bin unless used in interaction terms.





In [196]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler

# Fresh full-column scale for this check only -- dedup changed which side of the
# Item_Visibility/log_visibility and Calorie_Count_per_givenwt/log_Calorie pairs
# survived, so the scale_features list from step 7 no longer covers the right columns.
X_train_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

embedded_selector = SelectFromModel(LinearRegression())
embedded_selector.fit(X_train_scaled, y_train_series)

embedded_selected = X_train.columns[embedded_selector.get_support()].tolist()
print(f"Embedded check selected: {embedded_selected}")

Embedded check selected: ['Item_MRP', 'Outlet_Type']


In [197]:
min_agreement = 1  # 2 = only the strictest consensus features; 1 = also keep single-method signals

summary = anova_df.copy()
summary['Significant'] = summary['p_value'] < 0.05
summary['Embedded_Selected'] = summary['Feature'].isin(embedded_selected)
summary['Agreement'] = summary['Significant'].astype(int) + summary['Embedded_Selected'].astype(int)

summary = summary.sort_values(['Agreement', 'p_value'], ascending=[False, True]).reset_index(drop=True)

final_features = summary.loc[summary['Agreement'] >= min_agreement, 'Feature'].tolist()
print(f"Final feature set (statistically supported): {final_features}")

Final feature set (statistically supported): ['Outlet_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Item_Visibility', 'Item_Frequency', 'Item_Type', 'Item_Identifier_Type', 'Outlet_Status', 'Calorie_Count_per_givenwt']


In [198]:
summary

,Feature,Test,Statistic,p_value,Significant,Embedded_Selected,Agreement
0,Outlet_Type,ANOVA,694.230875,0.000000e+00,True,True,2
1,Item_MRP,Pearson,0.565303,0.000000e+00,True,True,2
2,Outlet_Identifier,ANOVA,232.590830,0.000000e+00,True,False,1
3,Outlet_Size,ANOVA,175.330323,5.614245e-75,True,False,1
4,Outlet_Location_Type,ANOVA,41.848905,8.628824e-19,True,False,1
5,Item_Visibility,Pearson,-0.087400,4.858708e-13,True,False,1
6,Item_Frequency,ANOVA,12.112705,5.039297e-04,True,False,1
7,Item_Type,ANOVA,2.149622,6.048495e-03,True,False,1
8,Item_Identifier_Type,ANOVA,4.917658,7.342226e-03,True,False,1
9,Outlet_Status,ANOVA,5.908955,1.508972e-02,True,False,1


### Observations:

 Agreement = 2 (both methods agree — keep, high confidence)
  - Outlet_Type and Item_MRP are the only features confirmed by both the univariate test and the embedded selector.
  These are your most trustworthy, non-redundant predictors — safe to anchor the model on them.

  Agreement = 1 (univariate-significant but embedded-rejected — the interesting group)
  - Nine features (Outlet_Identifier, Outlet_Size, Outlet_Location_Type, Item_Visibility, Item_Frequency, Item_Type,
  Item_Identifier_Type, Outlet_Status, Calorie_Count_per_givenwt) are statistically significant on their own but were
  not chosen by the embedded method.
  - This is exactly what was flagged earlier: ANOVA/Pearson test each feature in isolation, while the embedded method
  (Lasso/tree-importance, presumably) evaluates features jointly. The drop here strongly suggests
  redundancy/multicollinearity rather than these features being noise.
    - Outlet_Identifier dropping out while Outlet_Type survives confirms the earlier hypothesis — Identifier was just
  a finer-grained proxy for Type, and the embedded method correctly picked the more generalizable variable over the
  more granular one.
    - Outlet_Size, Outlet_Location_Type, Outlet_Status likely share the same fate — all outlet-level attributes that
  correlate with Outlet_Type, so once Outlet_Type is in the model, they add little marginal information.
    - Item_Visibility, Item_Frequency, Item_Type, Item_Identifier_Type, Calorie_Count_per_givenwt had weak effect
  sizes in the univariate test already (small F-stats/correlations) — the embedded method likely zeroed them out as
  not worth their model complexity cost once stronger features are present.

  Agreement = 0 (neither method selected — drop)
  - Item_Fat_Content and Weight_bin were rejected by both methods. This is the clearest signal in the table — no
  relationship with the target by any measure. Strong candidates to exclude entirely.

  Overall takeaway
  The two-method comparison cleans up the ambiguity from the univariate-only view: many features that looked
  "significant" alone are actually redundant once correlated outlet-level/item-level variables are considered
  jointly. For a lean, low-multicollinearity model, Outlet_Type + Item_MRP is the core; the Agreement=1 features are
  optional/marginal (worth testing incrementally, e.g. via ablation, since embedded selection can be sensitive to
  regularization strength); Agreement=0 features should be dropped.


In [200]:
print('MRP_bin' in X_train.columns)       # False if dedup is still in effect
print('MRP_bin' in anova_df['Feature'].values)   # False if the ANOVA refresh actually ran
print('MRP_bin' in final_features)        # tells you if the Agreement table cell was re-run


False
False
False


In [201]:
X_cv = pd.concat([X_train[final_features], X_test[final_features]])
y_cv = pd.concat([y_train,y_test])

X1=X_cv
y1= y_cv
XX = df_test[final_features]

X1.head()

,Outlet_Type,Item_MRP,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Item_Visibility,Item_Frequency,Item_Type,Item_Identifier_Type,Outlet_Status,Calorie_Count_per_givenwt
549,1,0.470709,9,1,2,0.035206,1,6,1,0,6.17500
7757,1,0.457877,7,0,1,0.047473,1,9,2,1,0.00000
764,1,-0.482625,8,0,2,0.076122,0,10,1,0,25.16800
6867,1,-1.603553,7,0,1,0.029845,1,6,1,1,5.41125
2716,1,0.218375,8,0,2,0.137228,1,13,1,0,61.03750


In [202]:
X_cv.to_csv('data/X_train_features.csv', index=False)
y_cv.to_csv('data/y_train.csv', index=False)
XX.to_csv('data/X_test_features.csv', index=False)

### Modelling Begins

In [203]:
"""5-fold CV model comparison and final model selection for BigMart sales prediction.

Assumes X_train, y_train, X_test are already loaded/preprocessed by the caller:
  - X_train, y_train: full labeled training data (pandas DataFrame / Series)
  - X_test: unlabeled holdout data to generate final predictions for (e.g. Kaggle test set)

X_train/y_train is split once into an 80% CV-train portion and a 20% holdout:
every model gets a 5-fold cross-validated mean +/- std (or out-of-fold RMSE) on the
CV-train portion, plus a single honest evaluation on the untouched holdout. The
holdout is never used for fitting or tuning -- only for evaluation. The final
submission model is refit on 100% of X_train/y_train before predicting X_test.

y_train must be passed in already log-transformed (log(Item_Outlet_Sales)) by the
caller -- raw sales are right-skewed (mean 2181 >> median 1794) and, without the
log-transform, squared-error loss underfits the high-value tail and nothing stops
predictions from going negative. main() exponentiates only the final X_test
predictions; every RMSE/MAE/R2 reported during comparison/tuning/CV is on the log
scale.
"""

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = seed
N_FOLDS = 5

In [204]:
def evaluate_regressor_kfold(model, X_train, y_train, model_name, folds=N_FOLDS):
    """Mean +/- std RMSE/MAE/R2 across k folds, plus out-of-fold RMSE, for one model."""
    model = clone(model)
    kf = KFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)

    fold_rmse, fold_mae, fold_r2 = [], [], []
    y_preds = pd.Series(index=y_train.index, dtype=float)

    for train_idx, val_idx in kf.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_fold_train, y_fold_train)
        pred = model.predict(X_fold_val)
        y_preds.iloc[val_idx] = pred

        fold_rmse.append(np.sqrt(mean_squared_error(y_fold_val, pred)))
        fold_mae.append(mean_absolute_error(y_fold_val, pred))
        fold_r2.append(r2_score(y_fold_val, pred))

    return {
        'Model': model_name,
        'RMSE_mean': round(float(np.mean(fold_rmse)), 3),
        'RMSE_std': round(float(np.std(fold_rmse)), 3),
        'MAE_mean': round(float(np.mean(fold_mae)), 3),
        'MAE_std': round(float(np.std(fold_mae)), 3),
        'R2_mean': round(float(np.mean(fold_r2)), 3),
        'R2_std': round(float(np.std(fold_r2)), 3),
        'OOF_RMSE': round(float(np.sqrt(mean_squared_error(y_train, y_preds))), 3),
    }

In [205]:
def evaluate_regressor_holdout(model, X_cv_train, y_cv_train, X_holdout, y_holdout, model_name):
    """Fit once on the CV-train split, score once on a held-out test split."""
    model = clone(model)
    model.fit(X_cv_train, y_cv_train)
    pred = model.predict(X_holdout)

    return {
        'Model': model_name,
        'Holdout_RMSE': round(float(np.sqrt(mean_squared_error(y_holdout, pred))), 3),
        'Holdout_MAE': round(float(mean_absolute_error(y_holdout, pred)), 3),
        'Holdout_R2': round(float(r2_score(y_holdout, pred)), 3),
    }


def build_candidate_models():
    return {
        'Linear':       Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
        'Ridge':        Pipeline([('scaler', StandardScaler()), ('model', Ridge())]),
        'Lasso':        Pipeline([('scaler', StandardScaler()), ('model', Lasso())]),
        'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_STATE),
        'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE, max_features=4),
        'Gradient':     GradientBoostingRegressor(random_state=RANDOM_STATE),
        'AdaBoost':     AdaBoostRegressor(random_state=RANDOM_STATE, n_estimators=10, learning_rate=0.1),
        'Bagging':      BaggingRegressor(
            GradientBoostingRegressor(random_state=RANDOM_STATE),
            random_state=RANDOM_STATE, n_estimators=5,
        ),
    }


def compare_models(X_cv_train, y_cv_train, X_holdout, y_holdout, folds=N_FOLDS):
    """For every candidate model: 5-fold CV on X_cv_train, plus a single
    held-out evaluation on X_holdout, combined into one row per model."""
    models = build_candidate_models()
    results = []
    for name, model in models.items():
        print(f"Comparing {name}")
        kfold_metrics = evaluate_regressor_kfold(model, X_cv_train, y_cv_train, name, folds)
        holdout_metrics = evaluate_regressor_holdout(model, X_cv_train, y_cv_train, X_holdout, y_holdout, name)
        results.append({**kfold_metrics, **holdout_metrics})
    return pd.DataFrame(results).sort_values('RMSE_mean').reset_index(drop=True)


def tune_gradient_boosting(X_train, y_train, folds=N_FOLDS):
    """Grid-search GradientBoosting hyperparameters with the same 5-fold CV used for comparison."""
    param_grid = {
        'max_features':      np.arange(1, 6, 1),
        'min_samples_leaf':  [0.1, 0.2, 0.3, 1, 2, 3, 4, 5],
        'max_leaf_nodes':    [2, 3, 4, 5, 6],
        'min_samples_split': [0.1, 0.2, 0.3, 0.4],
    }
    grid = GridSearchCV(
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        param_grid,
        cv=folds,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    best_rmse = np.sqrt(-grid.best_score_)
    return grid.best_estimator_, grid.best_params_, best_rmse


def build_final_model(tuned_gbr):
    return BaggingRegressor(tuned_gbr, random_state=RANDOM_STATE, n_estimators=5)

In [206]:
X_train = pd.read_csv('data/X_train_features.csv')
y_train = pd.read_csv('data/y_train.csv').squeeze('columns')
y_train = np.log(y_train)
X_test = pd.read_csv('data/X_test_features.csv')

In [207]:
X_cv_train, X_holdout, y_cv_train, y_holdout = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
)

In [208]:
print('Model comparison (5-fold CV on 80% + single holdout eval on 20%):')
comparison_df = compare_models(X_cv_train, y_cv_train, X_holdout, y_holdout)
comparison_df

Model comparison (5-fold CV on 80% + single holdout eval on 20%):
Comparing Linear
Comparing Ridge
Comparing Lasso
Comparing DecisionTree
Comparing RandomForest
Comparing Gradient
Comparing AdaBoost
Comparing Bagging


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,OOF_RMSE,Holdout_RMSE,Holdout_MAE,Holdout_R2
0,Bagging,0.523,0.015,0.405,0.010,0.735,0.018,0.524,0.520,0.401,0.737
1,Gradient,0.524,0.014,0.405,0.009,0.735,0.018,0.524,0.520,0.402,0.737
2,RandomForest,0.549,0.012,0.427,0.006,0.709,0.018,0.549,0.539,0.422,0.717
3,AdaBoost,0.560,0.015,0.436,0.009,0.697,0.019,0.560,0.560,0.435,0.695
4,Ridge,0.648,0.014,0.511,0.009,0.594,0.022,0.648,0.640,0.507,0.601
5,Linear,0.648,0.014,0.511,0.009,0.594,0.022,0.648,0.640,0.507,0.601
6,DecisionTree,0.752,0.013,0.580,0.013,0.453,0.032,0.752,0.741,0.576,0.467
7,Lasso,1.018,0.012,0.804,0.012,-0.001,0.001,1.018,1.014,0.798,-0.000


In [209]:
best_by_kfold = comparison_df.loc[comparison_df['RMSE_mean'].idxmin()]
best_by_holdout = comparison_df.loc[comparison_df['Holdout_RMSE'].idxmin()]
print(f"\nBest model by 5-fold CV RMSE: {best_by_kfold['Model']} (RMSE={best_by_kfold['RMSE_mean']})")
print(f"Best model by holdout RMSE:   {best_by_holdout['Model']} (RMSE={best_by_holdout['Holdout_RMSE']})")


Best model by 5-fold CV RMSE: Bagging (RMSE=0.523)
Best model by holdout RMSE:   Bagging (RMSE=0.52)


In [210]:
if best_by_kfold['Model'] == best_by_holdout['Model']:
    print(f"Both methods agree: {best_by_kfold['Model']} is the best model.")
else:
    print("Methods disagree on the best model -- the CV-best pick may not generalize as well "
          "as it looks; worth a closer look before committing to one.")

Both methods agree: Bagging is the best model.


In [211]:
print(f"tune gradient boosting ")
tuned_gbr, best_params, tuned_cv_rmse = tune_gradient_boosting(X_cv_train, y_cv_train)
print(f'\nTuned GradientBoosting params: {best_params}')
print(f'Tuned GradientBoosting 5-fold CV RMSE: {tuned_cv_rmse:.3f}')

tune gradient boosting 

Tuned GradientBoosting params: {'max_features': np.int64(5), 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'min_samples_split': 0.1}
Tuned GradientBoosting 5-fold CV RMSE: 0.523


In [212]:
final_model = build_final_model(tuned_gbr)
final_cv = evaluate_regressor_kfold(final_model, X_cv_train, y_cv_train, 'Final (Bagging + tuned GBR)')
final_cv

{'Model': 'Final (Bagging + tuned GBR)',
 'RMSE_mean': 0.524,
 'RMSE_std': 0.016,
 'MAE_mean': 0.405,
 'MAE_std': 0.01,
 'R2_mean': 0.735,
 'R2_std': 0.02,
 'OOF_RMSE': 0.524}

### Observations:

 Stability across folds
  - RMSE_std (0.016) and MAE_std (0.01) are both small relative to their means (~3% and ~2.5% respectively), and
  R²_std (0.02) is similarly tight around 0.735. This indicates the Bagging + tuned GBR ensemble generalizes
  consistently across folds rather than overfitting to particular splits — a good sign of a stable final model.

  OOF_RMSE matches RMSE_mean exactly (0.524 = 0.524)
  - This is a useful sanity check: the out-of-fold RMSE (computed from pooled OOF predictions) lines up with the
  average per-fold RMSE. If these diverged significantly, it would suggest fold-size imbalance or leakage; the exact
  match here confirms the CV estimate is a reliable, unbiased proxy for generalization performance.

  RMSE vs MAE gap (0.524 vs 0.405)
  - RMSE is moderately higher than MAE, as expected (RMSE penalizes large errors more). The gap (~0.12) suggests some
  larger residuals/outliers in predictions, but it's not an extreme gap — no sign of a small number of catastrophic
  mispredictions dominating the error.

  Scale of the errors (<1) strongly implies a log-transformed target
  - Raw Item_Outlet_Sales in this dataset ranges roughly 33–13,000+, so an RMSE of 0.524 only makes sense on a log
  scale. Worth back-transforming (e.g., np.expm1) predictions to the original sales scale when reporting error to
  stakeholders, since "RMSE = 0.524" is not business-interpretable on its own — the log-scale RMSE understates how
  large the error is on actual rupee/sales values, especially for high-revenue outlets.

  R² ≈ 0.735
  - This is a solid result for this dataset — BigMart sales prediction has substantial inherent noise (demand-driven randomness that no feature set fully captures), and public benchmarks on this dataset typically land in a similar range. Explaining ~73.5% of variance with a tight std (0.02) suggests the feature set (Outlet_Type, Item_MRP, etc. from your final selection) combined with the tuned ensemble is capturing the dominant signal well, without
  overfitting.

In [213]:
final_holdout = evaluate_regressor_holdout(
    final_model, X_cv_train, y_cv_train, X_holdout, y_holdout, 'Final (Bagging + tuned GBR)'
)
final_holdout

{'Model': 'Final (Bagging + tuned GBR)',
 'Holdout_RMSE': 0.519,
 'Holdout_MAE': 0.4,
 'Holdout_R2': 0.738}

### Observations:


Holdout closely tracks CV — strong confirmation of generalization
- All three holdout metrics fall comfortably within one CV std band of their CV means:
- RMSE: 0.519 vs CV 0.524 ± 0.016 → well inside range
- MAE: 0.400 vs CV 0.405 ± 0.010 → right at the edge, essentially identical
- R²: 0.738 vs CV 0.735 ± 0.02 → well inside range
- This is the result you want to see: the holdout set (never touched during CV/tuning) performs at least as well as
the cross-validated estimate, with differences small enough to be noise rather than a real effect.

No overfitting signal
- If holdout had performed notably worse than CV, that would flag tuning/feature-selection leakage (e.g., the
embedded selection or hyperparameter search peeking at validation folds). Instead it's marginally better across all
three metrics — reassuring, and consistent with normal sampling variation rather than anything systematic.

Practical conclusion
- CV and holdout now tell the same story, which means the reported performance (RMSE ≈ 0.52, MAE ≈ 0.40, R² ≈ 0.74
on the log scale) is a trustworthy estimate of how this model will perform on new data — not an artifact of a lucky
split or fold averaging.
- Same caveat as before applies: these are log-scale errors, so back-transform (np.expm1) before quoting RMSE/MAE
in original sales units if presenting to a non-technical audience.
- This is a reasonable point to call the modeling phase done — both validation strategies agree, so further
hyperparameter chasing is unlikely to yield a meaningfully different/better model.

In [214]:
print(f"\nFinal model 5-fold CV RMSE: {final_cv['RMSE_mean']} +/- {final_cv['RMSE_std']}")
print(f"Final model OOF RMSE: {final_cv['OOF_RMSE']}")
print(f"Final model holdout RMSE: {final_holdout['Holdout_RMSE']}, "
      f"MAE: {final_holdout['Holdout_MAE']}, R2: {final_holdout['Holdout_R2']}")


Final model 5-fold CV RMSE: 0.524 +/- 0.016
Final model OOF RMSE: 0.524
Final model holdout RMSE: 0.519, MAE: 0.4, R2: 0.738


In [215]:
# Refit on 100% of labeled data for the actual submission -- the holdout
# split above was only ever for evaluation, not for shrinking what the
# deployed model trains on.
print(f"Refit holdout")
final_model.fit(X_train, y_train)
# y_train is log(Item_Outlet_Sales) (caller's responsibility, see module
# docstring) -- invert here so the returned predictions are on the real
# sales scale, and are guaranteed positive.
predictions = np.exp(final_model.predict(X_test))

Refit holdout


In [216]:
predictions_df=pd.DataFrame(predictions, columns=[target])
predictions_df

,Item_Outlet_Sales
0,1382.794007
1,1158.842669
2,468.523142
3,2315.734253
4,5219.257358
...,...
5676,1881.701706
5677,2189.393503
5678,1668.443920
5679,3256.658009


In [217]:
print(f"predictions result ")
print(f"\nPrediction count: {len(predictions_df)} (X_test rows: {len(X_test)})")
print(f"Missing predictions: {predictions_df[target].isna().sum()}")
print(f"Negative predictions: {(predictions_df[target] < 0).sum()}")

predictions result 

Prediction count: 5681 (X_test rows: 5681)
Missing predictions: 0
Negative predictions: 0


In [218]:
print(predictions_df[target].describe())

count    5681.000000
mean     1944.473801
std      1186.946917
min        84.340592
25%       928.600096
50%      1841.089120
75%      2809.816179
max      5802.208547
Name: Item_Outlet_Sales, dtype: float64


### Conclusion:

Bagging (and Gradient, statistically indistinguishable from it) remains the best model by both evaluation methods; the tuning + bagging pipeline still buys nothing over the untuned default (0.523-0.524 across the board, all within noise of each other);
Lasso is reliably and systematically broken at this alpha, not randomly. Predictions are now well-formed (no negatives, no missing), with the one remaining known issue being the ~11% low-bias from the exp() retransformation — unrelated to Lasso, not something this run changes.
